f > f bag > z

In [ ]:
%load_ext autoreload
%autoreload 2

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import glob
import torch
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader

from data.dataset import TCGABags
from models.network import MonoCMIL
from core.losses import HypersphereLoss, Phase2Loss
from core.memory import FeatureMemoryBank

class Args:
    data_path = '/data3/jinsol/TCGA/features_conch_256/pt_files'
    lr = 0.001
    steps_p1 = 80
    steps_p2 = 120
    batch_size = 10

def apply_progressive_patch_masking(features, current_step, total_steps, max_mask_ratio=0.3):
    mask_ratio = max_mask_ratio * (current_step / total_steps)
    if mask_ratio <= 0.0:
        return features
        
    N = features.shape[1] if len(features.shape) == 3 else features.shape[0]
    num_keep = max(1, int(N * (1 - mask_ratio)))
    
    indices = torch.randperm(N)[:num_keep].to(features.device)
    if len(features.shape) == 3:
        return features[:, indices, :]
    return features[indices, :]

def custom_collate_fn(batch):
    features = [item[0] for item in batch]
    labels = torch.tensor([item[1] for item in batch])
    return features, labels

args = Args()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

incremental_classes = ['TCGA-BRCA', 'TCGA-LUAD', 'TCGA-COAD']
model = MonoCMIL(input_dim=512, hidden_dim=512, z_dim=256).to(device)

criterion_phase1 = HypersphereLoss().to(device)
criterion_phase2 = Phase2Loss().to(device)
memory_bank = FeatureMemoryBank(z_dim=256)

optimizer_phase1 = optim.Adam(model.abmil.parameters(), lr=args.lr)
optimizer_phase2 = optim.Adam(model.mlp_head.parameters(), lr=args.lr)

scheduler_phase1 = CosineAnnealingLR(optimizer_phase1, T_max=args.steps_p1)
scheduler_phase2 = CosineAnnealingLR(optimizer_phase2, T_max=args.steps_p2)

for task_id, cls_name in enumerate(incremental_classes):
    print("=" * 70)
    print(f"[Task {task_id}] '{cls_name}' 학습 시작")
    
    cls_files = glob.glob(os.path.join(args.data_path, f'{cls_name}/*.pt'))
    data_list = [(f, task_id) for f in cls_files]
    dataloader = DataLoader(TCGABags(data_list), batch_size=args.batch_size, shuffle=True, drop_last=True, collate_fn=custom_collate_fn)
    
    model.train()
    print(f"▶ Phase 1: ABMIL 최적화 ({args.steps_p1} steps)")
    
    data_iter = iter(dataloader)
    
    for step in range(args.steps_p1):
        optimizer_phase1.zero_grad()
        
        try:
            features_list, label = next(data_iter)
        except StopIteration:
            data_iter = iter(dataloader)
            features_list, label = next(data_iter)
            
        z_list = []
        for feat in features_list:
            feat = feat.to(device)
            feat = apply_progressive_patch_masking(feat, step, args.steps_p1, max_mask_ratio=0.3)
            z_s, _, _ = model(feat)
            z_list.append(z_s)
            
        z = torch.cat(z_list, dim=0)
        batch_labels = torch.full((z.shape[0],), task_id, dtype=torch.long).to(device)
        
        loss_p1, _ = criterion_phase1(z, batch_labels)
        loss_p1.backward()
        optimizer_phase1.step()
        scheduler_phase1.step()
        
        if (step + 1) % 20 == 0:
            print(f"  - Step [{step+1}/{args.steps_p1}] Hypersphere Loss: {loss_p1.item():.4f}")
            
    with torch.no_grad():
        clean_features_list, _ = next(iter(dataloader))
        f_bag_list = []
        for feat in clean_features_list:
            _, f_b, _ = model(feat.to(device))
            f_bag_list.append(f_b)
        clean_f_bag = torch.cat(f_bag_list, dim=0)
        
    memory_bank.update_statistics(task_id, clean_f_bag)
    current_anchor = memory_bank.generate_orthogonal_anchor(task_id, device)

    print(f"\n▶ Phase 2: MLP Head 최적화 (총 {args.steps_p2} steps)")
    
    for step in range(args.steps_p2):
        optimizer_phase2.zero_grad()
        
        current_stage = (step // 40) + 1 
        
        try:
            features_list, _ = next(data_iter)
        except StopIteration:
            data_iter = iter(dataloader)
            features_list, _ = next(data_iter)
            
        f_bag_list = []
        for feat in features_list:
            _, f_b, _ = model(feat.to(device))
            f_bag_list.append(f_b)
            
        f_bag = torch.cat(f_bag_list, dim=0).detach()
        z_cur = model.mlp_head(f_bag)
        
        f_past_dict = memory_bank.sample_past_features(task_id, batch_size=z_cur.shape[0], f_cur=f_bag)
        z_past_dict = {past_id: model.mlp_head(f_past.to(device)) for past_id, f_past in f_past_dict.items()}
        
        loss_p2 = criterion_phase2(z_cur, z_past_dict, current_anchor, stage_idx=current_stage, task_id=task_id)
        loss_p2.backward()
        optimizer_phase2.step()
        scheduler_phase2.step()
        
        if (step + 1) % 40 == 0:
            print(f"  - [Stage {current_stage} 완료] Step [{step+1}/{args.steps_p2}] Loss: {loss_p2.item():.4f}")

[Task 0] 'TCGA-BRCA' 학습 시작
▶ Phase 1: ABMIL 최적화 (80 steps)
  - Step [20/80] Hypersphere Loss: 0.0125
  - Step [40/80] Hypersphere Loss: 0.0123
  - Step [60/80] Hypersphere Loss: 0.0039
  - Step [80/80] Hypersphere Loss: 0.0024

▶ Phase 2: MLP Head 최적화 (총 120 steps)
  - [Stage 1 완료] Step [40/120] Loss: 0.1332
  - [Stage 2 완료] Step [80/120] Loss: 4.3512


NameError: name 'loss_stage3s' is not defined